In [ ]:
%load_ext autoreload

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

In [ ]:
%autoreload 2

from datetime import UTC, datetime
from pathlib import Path

import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from geopandas import gpd
from scipy.stats import pearsonr, spearmanr

from estuary.util.img import false_color
from estuary.util.plotting import plot_estuaries_ca

In [ ]:
FIG_DIR = Path("/Users/kyledorman/data/estuary/display/seasonal_analysis_v5/")
FIG_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
gdf = gpd.read_file("/Volumes/x10pro/estuary/geos/ca_data_w_empa_pmep_usgs.geojson")
gdf = gdf.rename(columns={"Site code": "region"})
skipped_regions = gdf[gdf.skipped].region
gdf = gdf[~gdf.skipped].copy()

gdf_points = gdf.copy()
gdf_points.geometry = gdf.to_crs("ESRI:54008").geometry.centroid.to_crs(gdf.crs)
gdf.head(2)

In [ ]:
preds_all_orig = pd.read_csv(
    "/Users/kyledorman/data/results/estuary/train/20260305-095554/merged_all_regions_timeseries_preds.csv"
)

preds_all_orig["acquired"] = pd.to_datetime(preds_all_orig["acquired"], errors="coerce", utc=True)
preds_all_orig["date"] = preds_all_orig.acquired.dt.date
preds_all_orig["month"] = preds_all_orig.acquired.dt.month
preds_all_orig["year"] = preds_all_orig.acquired.dt.year
preds_all = preds_all_orig.copy()
preds_all = (
    preds_all[(~preds_all.region.isin(skipped_regions)) & (preds_all.y_pred_unsure != 1)]
    .copy()
    .sort_values(by=["region", "acquired"])
    .reset_index(drop=True)
)

preds_all.head(3)

In [ ]:
# --- Binary model compatibility layer ---
# Expected mapping: y_pred in {0,1} with 0=closed, 1=open
# Ensure probability columns exist: p_closed, p_open

if "p_closed" not in preds_all.columns and "p_open" in preds_all.columns:
    preds_all["p_closed"] = 1.0 - preds_all["p_open"].astype(float)
if "p_open" not in preds_all.columns and "p_closed" in preds_all.columns:
    preds_all["p_open"] = 1.0 - preds_all["p_closed"].astype(float)

# # Optional legacy column (all zeros for a binary model)
# if "p_perched_open" not in preds_all.columns:
#     preds_all["p_perched_open"] = 0.0

preds_all["p_closed"] = preds_all["p_closed"].clip(0, 1)
preds_all["p_open"] = preds_all["p_open"].clip(0, 1)

preds_all.head(3)

In [ ]:
plot_estuaries_ca(
    gdf_points,
    markersize=10,
    jitter_deg=0.05,
    point_alpha=0.7,
    title="Dataset Estuaries",
    save_path=FIG_DIR / "state_map_all_estuaries.png",
)

In [ ]:
df = preds_all.copy()

samples_per_site = df.groupby("region").size().rename("n_samples")
days_per_site = df.assign(date=df["acquired"].dt.normalize()).groupby("region")["date"].nunique()
print(days_per_site.min(), days_per_site.max())

plt.figure(figsize=(6, 4))
plt.hist(samples_per_site, bins=15)
plt.xlabel("Number of samples per estuary")
plt.ylabel("Count of estuaries")
plt.title("Samples per estuary")
plt.tight_layout()
plt.show()

start = datetime(year=2018, month=1, day=1, tzinfo=UTC)
end = df["acquired"].max().normalize()
all_days = pd.date_range(start, end, freq="D")
df = df[df.acquired > start]
n_days_total = len(all_days)
days_per_site = df.assign(date=df["acquired"].dt.normalize()).groupby("region")["date"].nunique()

pct_days_per_site = 100 * days_per_site / n_days_total

plt.figure(figsize=(6, 4))
plt.hist(pct_days_per_site, bins=10)
plt.xlabel("Percent of days with valid samples")
plt.ylabel("Count of estuaries")
plt.title("Temporal coverage per estuary")
plt.tight_layout()
plt.show()

g = gdf_points[["region", "geometry"]].set_index("region").join(pct_days_per_site, how="left")
plot_estuaries_ca(
    g,
    column="date",
    cmap="viridis",
    title="Percent of days with valid samples",
    add_colorbar=True,
    show=True,
    markersize=20,
    jitter_deg=0.05,
    point_alpha=0.8,
    save_path=FIG_DIR / "state_map_pct_days_with_valid_sample.png",
)

In [ ]:
df = preds_all.copy()

df["year"] = df["acquired"].dt.year

df["year_month"] = df["acquired"].dt.to_period("M")

samples_by_year = df.groupby("year").size()

plt.figure(figsize=(6, 4))
samples_by_year.plot(kind="bar")
plt.xlabel("Year")
plt.ylabel("Number of samples")
plt.title("Sampling frequency by year")
plt.tight_layout()
plt.savefig(FIG_DIR / "per_year_sampling.png")
plt.show()


start = datetime(year=2018, month=1, day=1, tzinfo=UTC)
df = df[df.acquired > start]
samples_per_month = df.groupby("year_month").size().sort_index()

samples_per_month_dt = samples_per_month.copy()
samples_per_month_dt.index = samples_per_month_dt.index.to_timestamp()


fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(
    samples_per_month_dt.index,
    samples_per_month_dt.values,
    width=25,  # ~25 days looks good for monthly bars
)

ax.set_ylabel("Number of samples")
ax.set_title("Sampling frequency over time")

# Major ticks every year
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Minor ticks every 3 months (no labels)
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))

fig.autofmt_xdate()  # slight rotation if needed
plt.tight_layout()
plt.show()

df["month"] = df["acquired"].dt.month
samples_by_calendar_month = df.groupby("month").size()

plt.figure(figsize=(6, 4))
samples_by_calendar_month.plot(kind="bar")
plt.xlabel("Month")
plt.ylabel("Number of samples")
plt.title("Sampling frequency by calendar month")
plt.tight_layout()
plt.savefig(FIG_DIR / "per_month_sampling.png")
plt.show()

In [ ]:
capture_pct_by_cal_month = (
    100
    * samples_by_calendar_month
    / len(gdf)
    / 8
    / pd.Series([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
)

winter = capture_pct_by_cal_month[capture_pct_by_cal_month.index.isin([12, 1, 2])].mean()
spring = capture_pct_by_cal_month[capture_pct_by_cal_month.index.isin([3, 4, 5])].mean()
summer = capture_pct_by_cal_month[capture_pct_by_cal_month.index.isin([6, 7, 8])].mean()
fall = capture_pct_by_cal_month[capture_pct_by_cal_month.index.isin([9, 10, 11])].mean()

# (samples_by_calendar_month / samples_by_calendar_month.max()).round(3)
print("winter", round(winter, 1))
print("spring", round(spring, 1))
print("summer", round(summer, 1))
print("fall", round(fall, 1))

In [ ]:
df = preds_all.copy()
start = datetime(year=2018, month=1, day=1, tzinfo=UTC)
df = df[df.acquired > start]

times = pd.to_datetime(df["date"], utc=True, errors="coerce")
gaps_days = (
    df.assign(date=times)
    .dropna(subset=["date"])
    .sort_values(["region", "date"])
    .groupby("region")["date"]
    .diff()
    .dt.total_seconds()
    .div(86400.0)  # days as float
    .dropna()
)

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.hist(gaps_days, bins=60)
plt.xlabel("Gap (days)")
plt.ylabel("Count")
plt.yscale("log")
plt.title("All inter-sample gaps (within region)")
plt.tight_layout()
plt.savefig(FIG_DIR / "hist_all_gaps.png", dpi=300)
plt.show()

In [ ]:
def max_gap_days(times: pd.Series) -> float:
    times = times.sort_values()
    gaps = times.diff().dt.days.dropna()
    return gaps.max() if len(gaps) else 0


max_gap_per_site = df.groupby("region")["acquired"].apply(max_gap_days).rename("max_gap_days")

plt.figure(figsize=(6, 4))
plt.hist(max_gap_per_site, bins=15)
plt.xlabel("Longest gap between observations (days)")
plt.ylabel("Count of estuaries")
plt.title("Maximum observational gap per estuary")
plt.tight_layout()
plt.show()

g0 = gdf_points[["region", "geometry"]].set_index("region").join(pct_days_per_site, how="left")
g1 = gdf_points[["region", "geometry"]].set_index("region").join(max_gap_per_site, how="left")

g0["point_geom_lat"] = g0.geometry.centroid.y

plt.figure()
plt.scatter(
    g0["point_geom_lat"],
    g0["date"],
    s=40,
)

plt.xlabel("Latitude")
plt.ylabel("Pct time closed")

# optional trend line
z = np.polyfit(g0["point_geom_lat"], g0["date"], 1)
p = np.poly1d(z)
plt.plot(g0["point_geom_lat"], p(g0["point_geom_lat"]), lw=2)

plt.tight_layout()
plt.show()

# Pearson correlation (linear gradient)
r, p = pearsonr(g0["point_geom_lat"], g0["date"])
print("Pearson r:", round(r, 3), "p:", round(p, 3))

# Spearman correlation (monotonic trend)
rho, p = spearmanr(g0["point_geom_lat"], g0["date"])
print("Spearman rho:", round(rho, 3), "p:", round(p, 4))

import cartopy.crs as ccrs

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(10, 5),
    # constrained_layout=True,
    subplot_kw={"projection": ccrs.PlateCarree()},
)

plot_estuaries_ca(
    g0,
    column="date",
    cmap="viridis",
    title="Percent of days with valid samples",
    add_colorbar=True,
    show=False,
    markersize=20,
    jitter_deg=0.01,
    point_alpha=0.8,
    # save_path=FIG_DIR / "state_map_pct_days_with_valid_sample.png",
    ax=axes[0],
)

plot_estuaries_ca(
    g1,
    column="max_gap_days",
    cmap="viridis",
    title="Longest observation gap (days)",
    add_colorbar=True,
    show=False,
    markersize=20,
    jitter_deg=0.01,
    point_alpha=0.8,
    # save_path=FIG_DIR / "state_map_gap.png",
    ax=axes[1],
)

fig.tight_layout()
fig.savefig(FIG_DIR / "state_map_gap_pct_days_with_valid_sample.png")
plt.show()

In [ ]:
OPEN_THRESHOLD = 0.440

# 1) Make sure date is datetime64[ns]
df = preds_all.copy()
df["date"] = pd.to_datetime(df["date"])

# 2) Get global date range
start = datetime(year=2018, month=1, day=1)
# df["date"].min()
end = df["date"].max()

# 3) Build full MultiIndex: all regions × all dates
regions = df["region"].unique()
full_dates = pd.date_range(start, end, freq="D")

full_index = pd.MultiIndex.from_product(
    [regions, full_dates],
    names=["region", "date"],
)

# 4) Reindex and forward-fill within each region
full = df.set_index(["region", "date"]).sort_index().reindex(full_index)

# Forward-fill predictions within each region
full["y_pred"] = full["y_pred"].groupby(level="region").ffill()

prob_cols = ["p_closed", "p_open"]

for key in prob_cols:
    # Forward-fill predictions within each region
    full[key] = (
        full[key]
        .groupby(level="region")
        .apply(lambda s: s.interpolate(method="linear", limit_direction="both"))
        .ffill()
        .reset_index(level=0, drop=True)
    )

# normalize
P = full[prob_cols].to_numpy(dtype=float)
row_sum = np.nansum(P, axis=1, keepdims=True)

# Only normalize rows with valid positive mass
ok = np.isfinite(row_sum[:, 0]) & (row_sum[:, 0] > 0)
P_norm = P.copy()
P_norm[ok] = P[ok] / row_sum[ok]

# Write back
full.loc[:, prob_cols] = P_norm

region_min_date = df.groupby("region").date.min().max()
region_max_date = df.groupby("region").date.max().min()

full = full.reset_index()
full = full[full.date.between(region_min_date, region_max_date)].copy()
full = full.set_index(["region", "date"])

date_axis = full["p_open"].unstack("date").columns.to_list()
region_labels = full["p_open"].unstack("date").index.to_list()

probs = full[prob_cols].to_numpy(dtype=float)

y_pred = (full["p_open"] > OPEN_THRESHOLD).astype(int)

full["y_pred"] = y_pred

# Entropy
eps = 1e-12
P_clip = np.clip(probs, eps, 1.0)

entropy = -np.sum(P_clip * np.log(P_clip), axis=1)

# Normalize by log(K)
K = len(prob_cols)
entropy_norm = entropy / np.log(K)

# Add to full DataFrame
full["entropy"] = entropy
full["entropy_norm"] = entropy_norm

In [ ]:
P = preds_all[prob_cols].to_numpy(dtype=float)

bins = np.linspace(0, 1, 41)

fig, ax = plt.subplots(figsize=(6, 4))

for j, col in enumerate(prob_cols):
    vals = P[:, j]
    vals = vals[np.isfinite(vals)]
    ax.hist(vals, bins=bins, density=True, alpha=0.5, label=col)

ax.set_xlim(0, 1)
ax.set_xlabel("p(class)")
ax.set_ylabel("Density")
ax.set_title("Per-class probability distributions")
ax.legend()
plt.tight_layout()
plt.show()

fig, axs = plt.subplots(1, 2, figsize=(8, 3.5), sharex=True, sharey=True)

bins = np.linspace(0, 1, 41)

for ax, j, col in zip(axs, range(3), prob_cols):
    vals = P[:, j]
    vals = vals[np.isfinite(vals)]
    ax.hist(vals, bins=bins, density=True)
    ax.set_title(col)
    ax.set_xlim(0, 1)

axs[0].set_ylabel("Density")
for ax in axs:
    ax.set_xlabel("p(class)")

fig.suptitle("Per-class probability distributions", y=1.02)
plt.tight_layout()
plt.show()

pmax = np.nanmax(P, axis=1)
pmax = pmax[np.isfinite(pmax)]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(pmax, bins=np.linspace(0, 1, 41), density=True)
ax.set_xlim(0, 1)
ax.set_xlabel("max_k p(class_k)")
ax.set_ylabel("Density")
ax.set_title("Histogram of maximum class probability (model confidence)")
plt.tight_layout()
plt.show()

# Entropy
eps = 1e-12
P_clip = np.clip(P, eps, 1.0)

entropy = -np.sum(P_clip * np.log(P_clip), axis=1)

# Normalize by log(K)
K = len(prob_cols)
entropy_norm = entropy / np.log(K)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(entropy_norm, bins=40, density=True)
ax.set_xlabel("Normalized entropy")
ax.set_ylabel("Density")
ax.set_title("Distribution of 2-class prediction entropy")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
H_THR = 0.6

frac_high_entropy = (
    full["entropy_norm"]
    .groupby(level="region")
    .apply(lambda s: (s > H_THR).mean())
    .rename("frac_high_entropy")
)

g = gdf_points.set_index("region").join(frac_high_entropy, how="left")
plot_estuaries_ca(
    g,
    column="frac_high_entropy",
    cmap="viridis",
    title="Fraction High Entropy Days",
    markersize=20,
    jitter_deg=0.05,
    point_alpha=0.8,
    add_colorbar=True,
    show=True,
)

In [ ]:
def plot_timeseries(region, ax, legend=False):
    df = full.loc[region].sort_index()

    pcols = ["p_open", "p_closed"]
    df_roll = df[pcols].rolling("7D", min_periods=1).mean()

    ax.plot(df_roll.index, df_roll["p_open"], label="Open probability", lw=1.5)
    ax.plot(df_roll.index, df_roll["p_closed"], label="Closed probability", lw=1.5)

    dominant = df_roll.idxmax(axis=1)

    colors = {"p_open": "tab:blue", "p_closed": "tab:orange"}

    for col, color in colors.items():
        mask = dominant == col
        name = col.split("_")[1].capitalize()
        ax.fill_between(
            df_roll.index, 0, 1, where=mask, color=color, alpha=0.08, label=f"{name} State"
        )

    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel("7-day mean probability")
    ax.set_xlabel("Date")
    if legend:
        ax.legend(loc="lower left")
        ax.set_title("7-day mean probability")

In [ ]:
def plot_image(pth, ax):
    with rasterio.open(pth) as src:
        data = src.read().astype(np.float32)
        nodata = src.read(1, masked=True).mask
        img = false_color(data, nodata)

    ax.imshow(img)
    ax.axis("off")

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(15, 15), gridspec_kw={"width_ratios": [1, 2]})

region = 2145
name = gdf.set_index("region").loc[region]["Estuary_Name"]
plot_timeseries(region, axes[0, 1], legend=True)
pth = preds_all[(preds_all.region == region) & (preds_all.y_pred == 1)].iloc[100].source_tif
plot_image(pth, axes[0, 0])
axes[0, 0].set_title(f"{name}")

region = 51
name = gdf.set_index("region").loc[region]["Estuary_Name"]
plot_timeseries(region, axes[1, 1])
pth = preds_all[(preds_all.region == region) & (preds_all.y_pred == 1)].iloc[0].source_tif
plot_image(pth, axes[1, 0])
axes[1, 0].set_title(f"{name}")

region = 35
name = gdf.set_index("region").loc[region]["Estuary_Name"]
plot_timeseries(region, axes[2, 1])
pth = (
    preds_all[(preds_all.region == region) & (preds_all.y_pred == 1) & (preds_all.p_open < 0.9)]
    .iloc[1]
    .source_tif
)
plot_image(pth, axes[2, 0])
axes[2, 0].set_title(f"{name}")

plt.tight_layout()
plt.savefig(FIG_DIR / "time_series_example_6_panel.png", dpi=300)
plt.show()

In [ ]:
# compute pct_closed by region
pct_closed = full.groupby("region")["y_pred"].apply(lambda x: (x == 0).mean()).rename("pct_closed")

# join with estuary metadata
df_region = pct_closed.reset_index().merge(
    gdf[["region", "point_geom_lat"]], on="region", how="left"
)

In [ ]:
# Statewide fractions (mean probability over time) mapped by site

# mean_frac = full["p_open"].groupby(level="region").mean().rename("pct_open")
# g = gdf_points.set_index("region").join(mean_frac, how="left")
# plot_estuaries_ca(
#     g,
#     column="pct_open",
#     cmap="viridis",
#     title="Pct Time Open",
#     markersize=20,
#     jitter_deg=0.05,
#     point_alpha=0.8,
#     add_colorbar=True,
#     show=True,
#     save_path=FIG_DIR / "state_map_p_open.png",
# )

mean_frac = full["p_closed"].groupby(level="region").mean().rename("pct_closed")
g = gdf_points.set_index("region").join(mean_frac, how="left")
plot_estuaries_ca(
    g,
    column="pct_closed",
    cmap="viridis",
    title="Pct Time Closed",
    markersize=20,
    jitter_deg=0.05,
    point_alpha=0.8,
    add_colorbar=True,
    show=True,
    save_path=FIG_DIR / "state_map_p_closed.png",
)

import matplotlib.pyplot as plt

plt.figure(figsize=(5, 4))

plt.scatter(
    g["point_geom_lat"],
    g["pct_closed"],
    s=40,
)

plt.xlabel("Latitude")
plt.ylabel("Pct time closed")

# optional trend line
z = np.polyfit(g["point_geom_lat"], g["pct_closed"], 1)
p = np.poly1d(z)
plt.plot(g["point_geom_lat"], p(g["point_geom_lat"]), lw=2)

plt.tight_layout()
plt.show()


# Pearson correlation (linear gradient)
r, p = pearsonr(g["point_geom_lat"], g["pct_closed"])
print("Pearson r:", round(r, 3), "p:", round(p, 3))

# Spearman correlation (monotonic trend)
rho, p = spearmanr(g["point_geom_lat"], g["pct_closed"])
print("Spearman rho:", round(rho, 3), "p:", round(p, 3))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# -----------------------------------
# Monthly mean per estuary
# -----------------------------------
tmp = full.reset_index().copy()
tmp["date"] = pd.to_datetime(tmp["date"])
tmp["month"] = tmp["date"].dt.to_period("M").dt.to_timestamp()

region_month = tmp.groupby(["region", "month"])["p_open"].mean().reset_index()

# -----------------------------------
# Median + IQR across estuaries
# -----------------------------------
q = (
    region_month.groupby("month")["p_open"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
    .rename(columns={0.25: "q25", 0.5: "median", 0.75: "q75"})
    .sort_index()
)
# q = q.rolling(3, center=True, min_periods=1).mean()

# -----------------------------------
# Plot
# -----------------------------------
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    q.index,
    q["median"],
    color="tab:blue",
    lw=2,
    label="Median across estuaries",
)

ax.fill_between(
    q.index,
    q["q25"],
    q["q75"],
    color="tab:blue",
    alpha=0.25,
    label="IQR across estuaries",
)

ax.set_ylim(-0.05, 1.05)
ax.set_ylabel("Monthly mean p_open")
ax.set_xlabel("Date")
ax.set_title("Monthly openness by estuary: median and IQR across estuaries")
ax.legend(loc="lower left")
ax.grid(alpha=0.25)

plt.tight_layout()
# plt.savefig(FIG_DIR / "monthly_open_median_iqr_across_estuaries.png", dpi=200)
plt.show()

In [ ]:
tmp = full.reset_index().copy()
tmp["date"] = pd.to_datetime(tmp["date"])

# water year
tmp["water_year"] = tmp["date"].dt.year
tmp.loc[tmp["date"].dt.month >= 10, "water_year"] += 1
tmp = tmp[(tmp["water_year"] >= 2019) & (tmp["water_year"] <= 2025)]


# season mapping (water-year seasons)
def season(m):
    if m in [12, 1, 2]:
        return "Winter"
    if m in [3, 4, 5]:
        return "Spring"
    if m in [6, 7, 8]:
        return "Summer"
    return "Fall"


tmp["season"] = tmp["date"].dt.month.map(season)

region_season = tmp.groupby(["region", "water_year", "season"])["p_open"].mean().reset_index()
# region_season = region_season[region_season.water_year <= 2025]
# region_season = region_season[(region_season.water_year > 2018) | (region_season.season != "Winter")]

q = (
    region_season.groupby(["season", "water_year"])["p_open"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
    .rename(columns={0.25: "q25", 0.5: "median", 0.75: "q75"})
)

import matplotlib.pyplot as plt

seasons = ["Fall", "Winter", "Spring", "Summer"]

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey=True)

for ax, s in zip(axes.flatten(), seasons):
    d = q.loc[s]

    ax.plot(d.index, d["median"], lw=2, label="Median estuary")
    ax.fill_between(d.index, d["q25"], d["q75"], alpha=0.3, label="IQR")

    ax.set_title(s)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3)

axes[0, 0].set_ylabel("Mean p_open")
axes[1, 0].set_ylabel("Mean p_open")

axes[1, 0].set_xlabel("Water year")
axes[1, 1].set_xlabel("Water year")

fig.suptitle("Seasonal openness by water year\n(median and IQR across estuaries)")

axes[0, 0].legend()

plt.tight_layout()
fig.savefig(FIG_DIR / "yearly_trends_by_season.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

pcols = ["p_open", "p_closed"]

# --- 1) Mean across regions at each date ---
# If your index level is named "date", this will use it; otherwise it uses level=1.
date_level = "date" if "date" in full.index.names else 1

df_mean = full[pcols].groupby(level=date_level).mean().sort_index()

# --- 2) 14-day rolling mean on the mean time series ---
df_roll = df_mean.rolling("14D", min_periods=1).mean()

# --- 3) Plot ---
plt.figure(figsize=(10, 4))

plt.plot(df_roll.index, df_roll["p_open"], label="Open", lw=1.5)
# plt.plot(df_roll.index, df_roll["p_closed"], label="Closed", lw=1.5)

dominant = df_roll.idxmax(axis=1)

colors = {
    "p_open": "tab:blue",
    "p_closed": "tab:orange",
}

# for col, color in colors.items():
#     mask = dominant == col
#     cc = col.split("_")[1]

#     plt.fill_between(df_roll.index, 0, 1, where=mask, color=color, alpha=0.08, label=f"{cc} window")

plt.ylim(-0.05, 1.05)
plt.ylabel("14-day mean probability")
plt.xlabel("Date")
plt.legend(loc="lower left")
plt.title("14-day mean probability — mean across all regions")
plt.tight_layout()

plt.savefig(FIG_DIR / "all_regions_mean_timeseries.png")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap

# --- config ---
# map your class ids to names (edit if your ids differ)
class_names = {
    0: "Closed",
    1: "Open",
}

# choose colors for class ids 0/1/2 (edit to your preference)
# (example uses: 0=blue, 1=green, 2=orange)
cmap = ListedColormap(["tab:orange", "tab:blue"])
norm = BoundaryNorm(boundaries=[-0.5, 0.5, 2.5], ncolors=cmap.N)

# --- get (region, month) dominant class ---
# make sure we have region/date columns
tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])
tmp["month"] = tmp["date"].dt.to_period("M").dt.to_timestamp()

# mean p_closed per region
region_order = (
    full["p_closed"]
    .groupby(level="region")
    .mean()
    .sort_values(ascending=False)  # highest p_closed at top
    .index
)

# dominant class per (region, month): mode; tie-break by picking smallest id
dominant = (
    tmp.groupby(["region", "month"])["y_pred"]
    .agg(lambda x: x.value_counts().idxmax())
    .rename("dominant_class")
    .reset_index()
)

# pivot to estuary x month matrix
mat = dominant.pivot(index="region", columns="month", values="dominant_class").sort_index()
mat = mat.loc[region_order]

# --- plot ---
fig, ax = plt.subplots(figsize=(14, 8))

im = ax.imshow(mat.values, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")

# x-axis: months
months = mat.columns
ax.set_xticks(np.arange(len(months)))

# label every 4th month with a single letter (J/M/J/S…)
labels = [m.strftime("%b")[0] if (i % 4 == 0) else "" for i, m in enumerate(months)]
ax.set_xticklabels(labels)

# add vertical lines every 4th month
for i in range(0, len(months), 4):
    ax.axvline(i - 0.5, lw=0.6, alpha=0.4)

# y-axis: regions
ax.set_yticks(np.arange(len(mat.index)))
ax.set_yticklabels(mat.index)

ax.set_xlabel("Month")
ax.set_ylabel("Estuary (region)")
ax.set_title("Dominant predicted class by estuary and month")

# discrete colorbar with class names
cbar = fig.colorbar(im, ax=ax, ticks=[0, 1])
cbar.ax.set_yticklabels([class_names[i] for i in [0, 1]])

plt.tight_layout()
plt.savefig(FIG_DIR / "heatmap_estuary_month_dominant_class.png", dpi=200)
plt.show()

In [ ]:
tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])

# month-of-year, shifted so Oct=0, Nov=1, ..., Sep=11
tmp["wy_month"] = (tmp["date"].dt.month - 10) % 12

pcols = ["p_closed", "p_open"]

monthly_mean = tmp.groupby(["region", "wy_month"])[pcols].mean()

monthly_mean["dominant_class"] = monthly_mean[pcols].values.argmax(axis=1)
monthly_mean = monthly_mean.reset_index()

mat = monthly_mean.pivot(index="region", columns="wy_month", values="dominant_class")

region_order = full["p_closed"].groupby(level="region").mean().sort_values(ascending=False).index

mat = mat.loc[region_order]

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap

cmap = ListedColormap(
    [
        "tab:orange",  # 0 = Closed
        "tab:blue",  # 2 = Open
    ]
)
norm = BoundaryNorm([-0.5, 0.5, 2.5], cmap.N)

fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(mat.values, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")

# x-axis: water-year months
ax.set_xticks(np.arange(12))
ax.set_xticklabels(["O", "N", "D", "J", "F", "M", "A", "M", "J", "J", "A", "S"])

ax.set_yticks(np.arange(len(mat.index)))
ax.set_yticklabels(mat.index)

ax.set_xlabel("Water year month (Oct → Sep)")
ax.set_ylabel("Estuary (sorted by mean p_closed)")
ax.set_title("Mean seasonal dominant mouth state per estuary (water year)")

cbar = fig.colorbar(im, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(["Closed", "Perched Open", "Open"])

plt.tight_layout()
plt.savefig(FIG_DIR / "heatmap_estuary_water_year_month_mean_behavior.png", dpi=200)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

pcols = ["p_closed", "p_open"]

tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])
tmp["year"] = tmp["date"].dt.year

# Annual mean probability per estuary
annual = tmp.groupby(["region", "year"])[pcols].mean().reset_index()

stats = annual.groupby("region")[pcols].agg(["mean", "std"])

stats_long = (
    stats.stack(level=0)  # class
    .reset_index()
    .rename(columns={"level_1": "class", "mean": "mean_prob", "std": "interannual_std"})
)

import matplotlib.pyplot as plt

class_labels = {
    "p_closed": "Closed",
    "p_open": "Open",
}

colors = {
    "p_closed": "tab:orange",
    "p_open": "tab:blue",
}

plt.figure(figsize=(6, 5))

for cls in pcols:
    g = stats_long[stats_long["class"] == cls]
    plt.scatter(
        g["mean_prob"],
        g["interannual_std"],
        s=30,
        alpha=0.8,
        color=colors[cls],
        label=class_labels[cls],
    )

plt.xlabel("Mean annual probability")
plt.ylabel("Interannual variability (std of annual mean)")
plt.legend(title="Class")
plt.grid(alpha=0.3)
plt.tight_layout()

# plt.savefig(FIG_DIR / "interannual_variability_scatter_by_class.png", dpi=200)
plt.show()

In [ ]:
import pandas as pd

tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])

# Choose calendar year (swap to water year if you want)
tmp["year"] = tmp["date"].dt.year

# Annual mean P(open) per estuary
annual_open = tmp.groupby(["region", "year"])["p_open"].mean().reset_index()

# Interannual variability (std across years)
open_var = annual_open.groupby("region")["p_open"].std().rename("p_open_interannual_std")

g = gdf_points.set_index("region").join(open_var, how="left")

plot_estuaries_ca(
    g,
    column="p_open_interannual_std",
    cmap="viridis",
    title="Interannual variability of P(open)",
    markersize=20,
    jitter_deg=0.05,
    point_alpha=0.8,
    add_colorbar=True,
    show=True,
    save_path=FIG_DIR / "state_map_p_open_interannual_std.png",
)

In [ ]:
from io import StringIO

import pandas as pd

# curl https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/statewide/time-series/4/pcp/12/9/2017-2025.csv?base_prd=true

raw = """Date,Value,Anomaly
201809,15.82,-6.63
201909,28.37,5.92
202009,15.64,-6.81
202109,11.13,-11.32
202209,16.42,-6.03
202309,30.89,8.44
202409,22.97,0.52
202509,20.44,-2.01
"""

precip = pd.read_csv(StringIO(raw))

# Parse YYYYMM → datetime
precip["date"] = pd.to_datetime(precip["Date"].astype(str), format="%Y%m")

# Water year is just the year (since these are September-ending totals)
precip["water_year"] = precip["date"].dt.year

# Clean numeric columns
precip["precip_in"] = pd.to_numeric(precip["Value"], errors="coerce")
precip["anomaly_in"] = pd.to_numeric(precip["Anomaly"], errors="coerce")

# Final tidy table
precip = precip[["water_year", "precip_in", "anomaly_in"]].sort_values("water_year")

precip

In [ ]:
colors = ["tab:blue" if a >= 0 else "tab:orange" for a in precip["anomaly_in"]]

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(
    precip["water_year"],
    precip["precip_in"],
    color=colors,
    alpha=0.9,
)

ax.axhline(precip["precip_in"].mean(), linestyle="--", color="black", alpha=0.6)

ax.set_xlabel("Water Year")
ax.set_ylabel("Precipitation (inches)")
ax.set_title("California Water-Year Precipitation\n(Blue = Wet, Orange = Dry)")

ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "water_year_precip_bar.png", dpi=200)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

pcols = ["p_closed", "p_open"]

tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])

tmp["water_year"] = tmp["date"].dt.year + (tmp["date"].dt.month >= 10).astype(int)

# Statewide annual mean probability
statewide = tmp.groupby("water_year")[pcols].mean().sort_index()

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))

ax.stackplot(
    statewide.index,
    statewide["p_closed"],
    statewide["p_open"],
    labels=["Closed", "Open"],
    colors=["tab:orange", "tab:blue"],
    alpha=0.85,
)

ax.set_ylim(0, 1)
ax.set_ylabel("Mean statewide probability")
ax.set_xlabel("Water Year")
ax.legend(loc="upper right")
ax.set_title("Statewide Water Year mouth-state composition")

ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig(FIG_DIR / "statewide_water_year_composition_stackplot.png", dpi=200)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Join on overlapping water years
dfj = (
    statewide.reset_index()
    .merge(precip[["water_year", "precip_in"]], on="water_year", how="inner")
    .sort_values("water_year")
)

fig, ax = plt.subplots(figsize=(7, 5))

color_map = {
    "p_closed": "tab:orange",
    "p_open": "tab:blue",
}
label_map = {
    "p_closed": "Closed",
    "p_open": "Open",
}

for col in ["p_closed", "p_open"]:
    ax.scatter(
        dfj["precip_in"],
        dfj[col],
        s=60,
        alpha=0.85,
        color=color_map[col],
        label=label_map[col],
        edgecolor="black",
        linewidth=0.4,
    )

# Optional: label points with water year
for _, r in dfj.iterrows():
    ax.text(
        r["precip_in"] + 0.15,
        r["p_open"],  # anchor labels on one series to reduce clutter
        str(int(r["water_year"])),
        fontsize=8,
        alpha=0.8,
    )

ax.set_xlabel("CA Water-Year Precipitation (inches)")
ax.set_ylabel("Mean statewide probability")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend(loc="best", frameon=False)
ax.set_title("Statewide mouth-state probabilities vs precipitation")

plt.tight_layout()
plt.savefig(FIG_DIR / "pclass_vs_precip_scatter.png", dpi=200)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pcols = ["p_closed", "p_open"]
color_map = {"p_closed": "tab:orange", "p_open": "tab:blue"}
label_map = {"p_closed": "Closed", "p_open": "Open"}

# estuary-year means
tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])
tmp["water_year"] = tmp["date"].dt.year + (tmp["date"].dt.month >= 10).astype(int)

ey = (
    tmp.groupby(["region", "water_year"])[pcols]
    .mean()
    .reset_index()
    .merge(precip[["water_year", "precip_in"]], on="water_year", how="inner")
)


# summarize across estuaries within each WY
def q25(x):
    return np.nanpercentile(x, 25)


def q75(x):
    return np.nanpercentile(x, 75)


summ = (
    ey.groupby("water_year")
    .agg(
        precip_in=("precip_in", "first"),
        **{f"{c}_mean": (c, "mean") for c in pcols},
        **{f"{c}_q25": (c, q25) for c in pcols},
        **{f"{c}_q75": (c, q75) for c in pcols},
    )
    .reset_index()
    .sort_values("water_year")
)
import matplotlib.pyplot as plt
import numpy as np

pcols = ["p_closed", "p_open"]
color_map = {"p_closed": "tab:orange", "p_open": "tab:blue"}
label_map = {"p_closed": "Closed", "p_open": "Open"}

# fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(8, 10), sharex=True, sharey=True)

# for ax, c in zip(axes, pcols):
#     x = summ["precip_in"].to_numpy()
#     y = summ[f"{c}_mean"].to_numpy()
#     ylo = summ[f"{c}_q25"].to_numpy()
#     yhi = summ[f"{c}_q75"].to_numpy()
#     yerr = np.vstack([y - ylo, yhi - y])

#     ax.errorbar(
#         x, y, yerr=yerr,
#         fmt="o",
#         ms=7,
#         color=color_map[c],
#         ecolor=color_map[c],
#         elinewidth=2,
#         capsize=4,
#         alpha=0.9,
#     )

#     # annotate water year
#     for _, r in summ.iterrows():
#         ax.text(
#             r["precip_in"] + 0.2,
#             r[f"{c}_mean"],
#             str(int(r["water_year"])),
#             fontsize=8,
#             alpha=0.8,
#         )

#     ax.set_ylabel(label_map[c])
#     ax.set_ylim(0, 1)
#     ax.grid(True, alpha=0.3)

# axes[-1].set_xlabel("CA Water-Year Precipitation (inches)")

# fig.suptitle(
#     "Mouth-state probability vs precipitation\n(dot = mean across estuaries; bars = IQR across estuaries)",
#     y=0.98,
# )

# plt.tight_layout()
# plt.show()

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(8, 4), sharex=True, sharey=True)

for ax, c in zip(axes, pcols):
    x = summ["precip_in"].to_numpy()
    y = summ[f"{c}_mean"].to_numpy()
    ylo = summ[f"{c}_q25"].to_numpy()
    yhi = summ[f"{c}_q75"].to_numpy()
    yerr = np.vstack([y - ylo, yhi - y])

    # Scatter with IQR bars
    ax.errorbar(
        x,
        y,
        yerr=yerr,
        fmt="o",
        ms=7,
        color=color_map[c],
        ecolor=color_map[c],
        elinewidth=2,
        capsize=4,
        alpha=0.9,
    )

    # --- Line of best fit ---
    if len(x) > 1:
        m, b = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 100)
        ax.plot(xs, m * xs + b, color="black", linewidth=2, alpha=0.8, linestyle="--")

        # Optional: display slope
        ax.text(
            0.02,
            0.92,
            f"slope = {m:.3f}",
            transform=ax.transAxes,
            fontsize=9,
            verticalalignment="top",
        )

    # Annotate water year
    for _, r in summ.iterrows():
        ax.text(
            r["precip_in"] + 0.2,
            r[f"{c}_mean"],
            str(int(r["water_year"])),
            fontsize=8,
            alpha=0.8,
        )

    ax.set_ylabel(label_map[c])
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("CA Water-Year Precipitation (inches)")

fig.suptitle(
    "Mouth-state probability vs precipitation\n(dot = mean across estuaries; bars = IQR; black = linear fit)",
    y=0.98,
)

plt.tight_layout()
plt.savefig(FIG_DIR / "precip_vs_class_probs.png")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

color = "tab:blue"

# -----------------------------------
# Prepare data
# -----------------------------------
tmp = full.reset_index().copy()
tmp["date"] = pd.to_datetime(tmp["date"])
tmp["water_year"] = tmp["date"].dt.year + (tmp["date"].dt.month >= 10).astype(int)


def season_from_month(m):
    if m in [12, 1, 2]:
        return "Winter"
    if m in [3, 4, 5]:
        return "Spring"
    if m in [6, 7, 8]:
        return "Summer"
    return "Fall"


tmp["season"] = tmp["date"].dt.month.map(season_from_month)

# Keep only complete water years
tmp = tmp[(tmp["water_year"] >= 2019) & (tmp["water_year"] <= 2025)].copy()

# -----------------------------------
# Estuary-year-season means
# -----------------------------------
eys = (
    tmp.groupby(["region", "water_year", "season"])["p_open"]
    .mean()
    .reset_index()
    .merge(precip[["water_year", "precip_in"]], on="water_year", how="inner")
)


# -----------------------------------
# Summarize across estuaries within each WY × season
# -----------------------------------
def q25(x):
    return np.nanpercentile(x, 25)


def q75(x):
    return np.nanpercentile(x, 75)


summ = (
    eys.groupby(["season", "water_year"])
    .agg(
        precip_in=("precip_in", "first"),
        mean=("p_open", "mean"),
        q25=("p_open", q25),
        q75=("p_open", q75),
    )
    .reset_index()
)

# -----------------------------------
# Plot
# -----------------------------------
season_order = ["Fall", "Winter", "Spring", "Summer"]

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)

for ax, season in zip(axes.flatten(), season_order):
    d = summ[summ["season"] == season].sort_values("water_year").copy()

    x = d["precip_in"].to_numpy()
    y = d["mean"].to_numpy()
    ylo = d["q25"].to_numpy()
    yhi = d["q75"].to_numpy()
    yerr = np.vstack([y - ylo, yhi - y])

    # Scatter with IQR bars
    ax.errorbar(
        x,
        y,
        yerr=yerr,
        fmt="o",
        ms=8,
        color=color,
        ecolor=color,
        elinewidth=1.5,
        capsize=4,
        alpha=0.9,
    )

    # Linear fit
    if len(x) > 1 and np.isfinite(x).all() and np.isfinite(y).all():
        m, b = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 100)
        ax.plot(xs, m * xs + b, color="black", linewidth=2, alpha=0.8, linestyle="--")
    else:
        m = np.nan

    # Spearman correlation
    if len(d) > 1:
        rho, p = spearmanr(d["precip_in"], d["mean"])
    else:
        rho, p = np.nan, np.nan

    ax.text(
        0.03,
        0.95,
        f"slope = {m:.3f}\nrho = {rho:.2f}\np = {p:.2f}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
    )

    # Annotate water year
    for _, r in d.iterrows():
        ax.text(
            r["precip_in"] + 0.15,
            r["mean"],
            str(int(r["water_year"])),
            fontsize=8,
            alpha=0.8,
        )

    ax.set_title(season)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

axes[0, 0].set_ylabel("Open")
axes[1, 0].set_ylabel("Open")
axes[1, 0].set_xlabel("CA Water-Year Precipitation (inches)")
axes[1, 1].set_xlabel("CA Water-Year Precipitation (inches)")

fig.suptitle(
    "Seasonal openness vs water-year precipitation\n"
    "(dot = mean across estuaries; bars = IQR across estuaries; dashed line = linear fit)",
    y=0.98,
)

plt.tight_layout()
plt.savefig(FIG_DIR / "precip_vs_open_by_season.png", dpi=200)
plt.show()

In [ ]:
rs = []

for (season, region), d in eys.groupby(["season", "region"]):
    d = d.sort_values("water_year").copy()

    x = d["precip_in"].to_numpy() * 25.4 / 1000
    y = d["p_open"].to_numpy()

    # Linear fit
    if len(x) > 1 and np.isfinite(x).all() and np.isfinite(y).all():
        m, b = np.polyfit(x, y, 1)
        # xs = np.linspace(x.min(), x.max(), 100)
        # ax.plot(xs, m * xs + b, color="black", linewidth=2, alpha=0.8, linestyle="--")
    else:
        m = np.nan

    rs.append(
        {
            "region": region,
            "season": season,
            "slope": m,
            "p_open": y.mean(),
        }
    )

rs_df = pd.DataFrame(rs)

import numpy as np
from sklearn.linear_model import TheilSenRegressor

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(7, 7), sharex=True, sharey=True)

for ax, season in zip(axes.flatten(), ["Winter", "Spring", "Fall", "Summer"]):
    season_df = rs_df[rs_df.season == season].set_index("region")

    g = gdf_points.set_index("region").join(season_df, how="left")

    g["latitude"] = g.geometry.centroid.y
    ax.set_title(season)
    ax.scatter(x=g.slope, y=g.latitude)

    x = g.slope
    y = g.latitude

    # Linear fit
    if len(x) > 1 and np.isfinite(x).all() and np.isfinite(y).all():
        X = np.array(x).reshape(-1, 1)
        y = np.array(y)

        model = TheilSenRegressor()
        model.fit(X, y)

        m = model.coef_[0]
        b = model.intercept_
        xs = np.linspace(x.min(), x.max(), 100)
        ax.plot(xs, m * xs + b, color="black", linewidth=2, alpha=0.8, linestyle="--")
    else:
        m = np.nan

    # Spearman correlation
    if len(d) > 1:
        rho, p = spearmanr(x, y)
    else:
        rho, p = np.nan, np.nan

    ax.text(
        0.67,
        0.95,
        f"slope = {m:.3f}\nrho = {rho:.2f}\np = {p:.2f}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
    )

axes[0, 0].set_ylabel("latitude")
axes[1, 0].set_ylabel("latitude")
axes[1, 0].set_xlabel("slope (p_open/precip)")
axes[1, 1].set_xlabel("slope (p_open/precip)")

fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

pcols = ["p_closed", "p_open"]

tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])
tmp["year"] = tmp["date"].dt.year  # swap to water year if desired

annual = tmp.groupby(["region", "year"])[pcols].mean().reset_index()

# std of annual means per class
interannual_std = annual.groupby("region")[pcols].std()

# overall variability score per estuary
interannual_std["mean_variability"] = interannual_std.mean(axis=1)

# sort
interannual_std = interannual_std.sort_values("mean_variability", ascending=False)

g = gdf_points.merge(interannual_std, on="region", how="left")

plot_estuaries_ca(
    g,
    column="mean_variability",
    # cmap="brg",
    title="Mean Variablility",
    markersize=20,
    jitter_deg=0.05,
    point_alpha=0.8,
    add_colorbar=True,
    show=True,
    # save_path=FIG_DIR / "state_map_dominant_class.png"
)

top_n = 30  # adjust

plot_df = interannual_std.head(top_n)

plt.figure(figsize=(6, 8))

dominant = annual.groupby("region")[pcols].mean().idxmax(axis=1)

colors = {
    "p_closed": "tab:orange",
    "p_open": "tab:blue",
}

bar_colors = dominant.loc[plot_df.index].map(colors)

plt.barh(plot_df.index.astype(str), plot_df["mean_variability"], color=bar_colors)

plt.gca().invert_yaxis()
plt.xlabel("Interannual variability\n(std of annual mean probability)")
plt.ylabel("Estuary (region)")
plt.title("Most year-to-year variable estuaries")

plt.grid(axis="x", alpha=0.3)
plt.tight_layout()

plt.savefig(FIG_DIR / "most_variable_estuaries_bar.png", dpi=200)
plt.show()

In [ ]:
tmp = full.reset_index().copy()
tmp["date"] = pd.to_datetime(tmp["date"])
tmp["wy_month"] = (tmp["date"].dt.month - 10) % 12

region_month = tmp.groupby(["region", "wy_month"])["p_open"].mean().reset_index()

q = (
    region_month.groupby("wy_month")["p_open"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
    .rename(columns={0.25: "q25", 0.5: "median", 0.75: "q75"})
)

months = np.arange(12)
month_labels = ["O", "N", "D", "J", "F", "M", "A", "M", "J", "J", "A", "S"]

fig, ax = plt.subplots(figsize=(9, 3))

ax.plot(months, q["median"], lw=2, color="tab:blue", label="Region median")
ax.fill_between(months, q["q25"], q["q75"], color="tab:blue", alpha=0.25, label="Region IQR")

ax.set_ylim(-0.05, 1.05)
ax.set_xticks(months)
ax.set_xticklabels(month_labels)
ax.set_xlabel("Water year month (Oct → Sep)")
ax.set_ylabel("Mean openness probability")
ax.set_title("Statewide seasonal openness across regions")
ax.grid(alpha=0.25)
ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig(FIG_DIR / "statewide_seasonal_openness.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

pcols = ["p_closed", "p_open"]

tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])
tmp["year"] = tmp["date"].dt.year

# Expected days = sum of probabilities within each (region, year)
expected_days = tmp.groupby(["region", "year"])[pcols].sum().reset_index()

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, constrained_layout=True)

labels = {
    "p_closed": "Closed",
    "p_open": "Open",
}

colors = {
    "p_closed": "tab:orange",
    "p_open": "tab:blue",
}

bins = np.linspace(0, 365, 31)  # ~12-day bins

for ax, p in zip(axes, pcols):
    ax.hist(expected_days[p], bins=bins, color=colors[p], alpha=0.8, edgecolor="black")
    ax.set_ylabel("Count")
    ax.set_title(labels[p], loc="left", fontsize=11, weight="bold")
    ax.grid(axis="y", alpha=0.25)

axes[-1].set_xlabel("Expected days per year")

plt.suptitle(
    "Distribution of expected days per year in each mouth state\n(across all regions and years)",
    y=1.05,
)

# plt.savefig(FIG_DIR / "hist_expected_days_per_year_by_class.png", dpi=200)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

pcols = ["p_closed", "p_open"]

STATE_MAP = {
    0: "closed",
    1: "open",
}

N = 4  # days

tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])

# dominant state
tmp["state"] = tmp[pcols].values.argmax(axis=1)
tmp["state_name"] = tmp["state"].map(STATE_MAP)


def get_runs(df):
    df = df.sort_values("date").copy()
    df["run_id"] = (df["state"] != df["state"].shift()).cumsum()

    runs = (
        df.groupby("run_id")
        .agg(
            state=("state_name", "first"),
            start=("date", "first"),
            end=("date", "last"),
            duration=("date", "size"),
        )
        .reset_index(drop=True)
    )

    # carry region through (df has a single region)
    runs["region"] = df["region"].iloc[0]
    return runs[["region", "state", "start", "end", "duration"]]


runs = tmp.groupby("region", group_keys=False).apply(get_runs).reset_index(drop=True)

runs

In [ ]:
import numpy as np
import pandas as pd

pcols = ["p_closed", "p_open"]
state_labels = {
    0: "Closed",
    1: "Open",
}

tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])


def state_durations(df):
    df = df.sort_values("date").copy()

    # identify change points
    df["new_run"] = (df["y_pred"] != df["y_pred"].shift()).cumsum()

    runs = df.groupby(["new_run", "y_pred"]).size().reset_index(name="duration_days")
    return runs[["y_pred", "duration_days"]]


durations = tmp.groupby("region", group_keys=False).apply(state_durations).reset_index(drop=True)

durations["state_name"] = durations["y_pred"].map(state_labels)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True, constrained_layout=True)

colors = {
    "Closed": "tab:orange",
    "Open": "tab:blue",
}

bins = np.logspace(0, 3, 40)  # 1 to 1000 days

for ax, state in zip(axes, ["Closed", "Open"]):
    data = durations.loc[durations["state_name"] == state, "duration_days"]

    ax.hist(data, bins=bins, color=colors[state], alpha=0.85)

    ax.set_xscale("log")
    ax.set_ylabel("Count")
    ax.set_title(state, loc="left", fontsize=11, weight="bold")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("State duration (days, log scale)")

plt.suptitle("Distribution of dominant-state durations\n(across all estuaries)", y=1.02)

plt.savefig(FIG_DIR / "hist_state_duration_log.png", dpi=200)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

pcols = ["p_closed", "p_open"]


def entropy_row(P):
    P = np.clip(P, 1e-6, 1)
    return -(P * np.log(P)).sum(axis=1)


tmp = full.reset_index()
tmp["date"] = pd.to_datetime(tmp["date"])

# entropy
tmp["entropy"] = entropy_row(tmp[pcols].values)

# dominant state
tmp["state"] = tmp["y_pred"]

# transitions per region
tmp["transition"] = tmp.groupby("region")["state"].transform(lambda x: (x != x.shift()).astype(int))

# per-region metrics
metrics = tmp.groupby("region").agg(
    f_closed=("p_closed", "mean"),
    f_open=("p_open", "mean"),
    entropy_mean=("entropy", "mean"),
    transitions=("transition", "sum"),
    n_days=("date", "size"),
)

metrics["transitions_per_year"] = metrics["transitions"] / (metrics["n_days"] / 365)


def mean_persistence_days(states):
    runs = (states != states.shift()).cumsum()
    return runs.value_counts().mean()


persistence = (
    tmp.groupby("region")["state"].apply(mean_persistence_days).rename("mean_persistence_days")
)

metrics = metrics.join(persistence)

# def assign_regime_refined(r):
#     # Layer 1: dominance flags
#     if r.f_closed >= 0.8:
#         return "Mostly closed"

#     if r.f_open >= 0.8:
#         return "Mostly open"

#     # Layer 2: dynamic regimes
#         return "Perched-limited"

#     if r.transitions_per_year >= 9:
#         return "Transitional"


def assign_regime_refined_row(r):
    # dominance flags
    if r.f_closed >= 0.8:
        return "Mostly closed"
    if r.f_open >= 0.8:
        return "Mostly open"

        # dynamic regimes
        return "Perched-limited"
        return "Transitional"

    return "Other dynamic"


metrics["regime"] = metrics.apply(assign_regime_refined_row, axis=1)

metrics

In [ ]:
metrics.sort_values("entropy_mean").tail(10)

In [ ]:
def filter_outliers(df: pd.DataFrame, depth_col: str = "height") -> pd.DataFrame:
    # Quick-and-dirty outlier removal on depth values using IQR
    y = df[depth_col].astype(float)
    q1 = y.quantile(0.25)
    q3 = y.quantile(0.75)
    iqr = q3 - q1

    # Keep points within 3 * IQR (lenient to preserve events)
    lo = q1 - 3 * iqr
    hi = q3 + 3 * iqr

    mask = (y >= lo) & (y <= hi)
    return df.loc[mask].copy()


def contiguous_segments(
    df: pd.DataFrame, time_col: str = "acquired", gap_hours: int = 48
) -> list[pd.DataFrame]:
    """
    Find all contiguous segments in a time series (gaps > gap_hours separate segments).
    """
    out = []

    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce", utc=True)
    df = df.dropna(subset=[time_col]).sort_values(time_col)
    if df.empty:
        return []

    gap = pd.Timedelta(hours=gap_hours)
    seg_id = (df[time_col].diff() > gap).cumsum()
    # Find all segments
    for sid in seg_id.unique():
        seg = df[seg_id == sid]

        if seg.empty:
            continue

        seg = filter_outliers(seg.copy())
        out.append(seg.reset_index(drop=True))

    return out


def subsample_time(df, resample_amount: str = "30min", columns: list[str] | None = None):
    df["timestamp_utc"] = pd.to_datetime(
        df["timestamp_utc"],
        utc=True,
        errors="coerce",
    )
    df = df.dropna(subset=["timestamp_utc"])
    if columns is not None:
        columns.append("timestamp_utc")
        df = df[columns]
    return (
        df.set_index("timestamp_utc")
        .sort_index()
        .resample(resample_amount, origin="epoch")
        .median()  # or .median(), .first(), etc.
        .dropna()
        .reset_index()
    )


def inlet_only_tidal_attenuation(
    s: pd.Series,
    *,
    window_days: int = 3,
    tz: str = "UTC",
) -> pd.DataFrame:
    """
    Inlet-only tidal attenuation (suppression index).

    Returns DataFrame with:
        - daily_range
        - tidal_range_proxy (rolling min of daily_range)
        - attenuation_pct
    """

    if not isinstance(s.index, pd.DatetimeIndex):
        raise TypeError("Series must have DatetimeIndex")

    s = s.copy()

    s = s.sort_index().dropna()

    # Daily min/max
    daily = s.resample("1D").agg(["min", "max"])
    daily.columns = ["daily_min", "daily_max"]

    # Daily range
    daily["daily_range"] = daily["daily_max"] - daily["daily_min"]

    # Rolling minimum (tidal proxy)
    daily["tidal_range_proxy"] = (
        daily["daily_range"].rolling(window_days, min_periods=1, center=True).min()
    )

    return daily


def calc_tidal_attenuation(
    s: pd.Series,
    o: pd.Series,
    *,
    window_days: int = 3,
) -> pd.DataFrame:
    if not isinstance(s.index, pd.DatetimeIndex):
        raise TypeError("Series must have DatetimeIndex")
    if not isinstance(o.index, pd.DatetimeIndex):
        raise TypeError("Series must have DatetimeIndex")

    s = s.sort_index().dropna()
    o = o.sort_index().dropna()

    # Daily min/max + range (peak-to-trough)
    sd = s.resample("1D").agg(["min", "max"])
    sd.columns = ["s_daily_min", "s_daily_max"]
    sd["s_daily_range"] = sd["s_daily_max"] - sd["s_daily_min"]

    od = o.resample("1D").agg(["min", "max"])
    od.columns = ["o_daily_min", "o_daily_max"]
    od["o_daily_range"] = od["o_daily_max"] - od["o_daily_min"]

    # Rolling min range
    s_range = sd["s_daily_range"].rolling(window_days, min_periods=1, center=True).min()

    # Align indices (in case s/o cover different date spans)
    s_range, o_range = s_range.align(od["o_daily_range"], join="inner")

    ratio = s_range / np.maximum(eps, o_range)

    out = pd.DataFrame(index=s_range.index)
    out["estuery_daily_range"] = s_range
    out["tide_daily_range"] = o_range
    out["tidal_ratio"] = ratio.clip(lower=0.0, upper=1.0)
    out["tidal_attenuation"] = 1 - out["tidal_ratio"]

    return out

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import welch


def rolling_tidal_spectral_metrics_welch(
    s: pd.Series,
    *,
    freq: str = "30min",
    window: str = "3D",
    band_hours: tuple[float, float] = (10, 14),
    min_frac: float = 0.8,
    detrend: str = "constant",
    nperseg_max: int = 512,
    max_interp_gap: str | None = "2h",
) -> pd.DataFrame:
    """
    Rolling spectral metrics using Welch PSD on a regular grid.

    Returns DataFrame with columns:
      - tp_frac
      - band_peakiness_vs_outband_median
      - band_peak_period_hours
      - band_peak_freq_cph
      - n_valid
    """
    if not isinstance(s.index, pd.DatetimeIndex):
        raise TypeError("s must have a DatetimeIndex")

    # Ensure UTC
    s = s.copy()
    if s.index.tz is None:
        s.index = s.index.tz_localize("UTC")
    else:
        s.index = s.index.tz_convert("UTC")
    s = s.sort_index()

    # Regularize to fixed cadence
    s = s.resample(freq).mean()

    freq_td = pd.Timedelta(freq)
    window_td = pd.Timedelta(window)
    n_expected = int(window_td / freq_td) + 1
    min_n = int(np.ceil(n_expected * min_frac))

    dt_hours = freq_td.total_seconds() / 3600.0
    fs = 1.0 / dt_hours  # samples per hour

    # Band in cycles/hour
    f_low = 1.0 / band_hours[1]
    f_high = 1.0 / band_hours[0]

    # Optional interpolation limit
    interp_limit = None
    if max_interp_gap is not None:
        interp_limit = int(pd.Timedelta(max_interp_gap) / freq_td)

    out_rows = []

    for t in s.index:
        seg = s.loc[t - window_td : t]
        n_valid = int(seg.notna().sum())

        if n_valid < min_n:
            out_rows.append((np.nan, np.nan, np.nan, np.nan, n_valid))
            continue

        seg2 = seg.copy()

        # Interpolate small gaps only (keeps FFT assumptions reasonable)
        if interp_limit is not None:
            seg2 = seg2.interpolate(limit_area="inside", limit=interp_limit)

        # If still missing, skip
        if seg2.isna().any():
            out_rows.append((np.nan, np.nan, np.nan, np.nan, n_valid))
            continue

        x = seg2.to_numpy(dtype=float)

        # Welch PSD
        nperseg = min(len(x), nperseg_max)
        f, Pxx = welch(
            x,
            fs=fs,
            nperseg=nperseg,
            detrend=detrend,
            scaling="density",
        )

        if len(Pxx) == 0:
            out_rows.append((np.nan, np.nan, np.nan, np.nan, n_valid))
            continue

        Pxx = Pxx.copy()
        Pxx[0] = 0.0  # remove DC

        # Total power over all positive freqs
        total = np.trapezoid(Pxx, f)
        if not np.isfinite(total) or total <= 0:
            out_rows.append((np.nan, np.nan, np.nan, np.nan, n_valid))
            continue

        band = (f >= f_low) & (f <= f_high)
        if not band.any():
            out_rows.append((np.nan, np.nan, np.nan, np.nan, n_valid))
            continue

        band_pow = np.trapezoid(Pxx[band], f[band])
        tp_frac = float(band_pow / total)

        # Peak within band
        band_vals = Pxx[band]
        peak_val = float(np.max(band_vals))
        # med_in = float(np.median(band_vals))
        # mean_in = float(np.mean(band_vals))

        # Outside-band median (exclude DC already, but also exclude the band)
        outband = (~band) & (f > 0)
        outband_vals = Pxx[outband]
        med_out = float(np.median(outband_vals)) if outband_vals.size else np.nan

        eps = 1e-12
        # peakiness_med = peak_val / max(med_in, eps)
        # peakiness_mean = peak_val / max(mean_in, eps)
        peakiness_vs_out = peak_val / max(med_out, eps) if np.isfinite(med_out) else np.nan

        # Where is the peak (nice sanity check)
        peak_idx = int(np.argmax(band_vals))
        f_band = f[band]
        f_peak = float(f_band[peak_idx])  # cycles per hour
        period_peak = float(1.0 / f_peak) if f_peak > 0 else np.nan

        out_rows.append((tp_frac, peakiness_vs_out, period_peak, f_peak, n_valid))

    out = pd.DataFrame(
        out_rows,
        index=s.index,
        columns=[
            "tp_frac",
            "band_peakiness_vs_outband_median",
            "band_peak_period_hours",
            "band_peak_freq_cph",
            "n_valid",
        ],
    )

    return out

In [ ]:
tidal_data = np.load("/Volumes/x10pro/estuary/geos/region_tide_heights.npz", allow_pickle=True)
tide_minutes = tidal_data["minutes"]
tide_minutes = pd.to_datetime(tide_minutes, utc=True)
tidal_sites = list(map(int, tidal_data["site_ids"].tolist()))


def region_tidal_data(region, start, end):
    idx = tidal_sites.index(region)
    tide_df = (
        pd.DataFrame(
            {
                "timestamp_utc": pd.to_datetime(tide_minutes, utc=True),
                "tide_height": tidal_data["elev"][idx],
            }
        )
        .set_index("timestamp_utc")
        .sort_index()
    )

    return tide_df.loc[start:end].reset_index()


list(tidal_data.keys())

In [ ]:
import tqdm

water_data_path = Path("/Volumes/x10pro/estuary/water_data/processed/")

all_segments = {}
subsample = "30min"
max_gap = 8


def get_region_tide_series(sdf: pd.DataFrame, region, subsample) -> pd.Series:
    ocean = region_tidal_data(region, sdf.index.min(), sdf.index.max())
    ocean = ocean.dropna(subset=["timestamp_utc"]).sort_values("timestamp_utc")
    ocean = subsample_time(ocean, subsample, ["tide_height"])
    return ocean.set_index("timestamp_utc")["tide_height"]


for region, _ in tqdm.tqdm(gdf.set_index("region").iterrows(), total=len(gdf)):
    region_water_data_dir = water_data_path / str(region)
    if not region_water_data_dir.exists():
        continue

    region_segments = []
    for p in region_water_data_dir.glob("*.csv"):
        df = pd.read_csv(p, dtype={"sensor_id": "string", "source": "string"})
        for _, sdf in df.groupby(["sensor_id", "source"]):
            segments = contiguous_segments(sdf, time_col="timestamp_utc", gap_hours=max_gap)
            segments = [subsample_time(seg, subsample, ["height"]) for seg in segments]
            region_segments.extend(segments)

    if len(region_segments) == 0:
        continue

    region_segments = sorted(
        region_segments,
        key=lambda sdf: sdf.timestamp_utc.max() - sdf.timestamp_utc.min(),
        reverse=True,
    )

    all_median = pd.concat(
        [subsample_time(s, subsample, ["height"]) for s in region_segments]
    ).height.median()
    meds = [s.height.median() for s in region_segments]
    keep = [np.abs(m - all_median) < 3 for m in meds]
    region_segments = [s for s, k in zip(region_segments, keep) if k]

    seg = pd.concat(region_segments)
    df = seg.drop_duplicates("timestamp_utc").sort_values("timestamp_utc")

    # parse timestamps + set index (UTC)
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp_utc"])
    df = df.set_index("timestamp_utc").sort_index()

    # z-score (if you want it)
    df["height_z"] = (df["height"] - df["height"].mean()) / df["height"].std()

    region_segments = contiguous_segments(
        df.reset_index(), time_col="timestamp_utc", gap_hours=max_gap
    )
    region_segments = [
        s
        for s in region_segments
        if (s.timestamp_utc.max() - s.timestamp_utc.min()) > pd.Timedelta("7D")
    ]

    out_segments = []
    for sdf in region_segments:
        sdf = sdf.set_index("timestamp_utc")
        s = sdf.height
        tide = get_region_tide_series(s, region, subsample)

        tidal_attn = calc_tidal_attenuation(s, tide, window_days=5)
        tidal_hf = tidal_attn.reindex(sdf.index, method="ffill")

        # m12 = rolling_tidal_spectral_metrics_welch(
        #     s,
        #     band_hours=(10, 14),
        # )

        # m24 = rolling_tidal_spectral_metrics_welch(
        #     s,
        #     band_hours=(20, 28),
        # )

        # tp_total = m12.tp_frac + m24.tp_frac
        # tp_hf = tp_total.reindex(sdf.index, method="ffill")

        # tidal_attn_prox = inlet_only_tidal_attenuation(s)
        # tidal_prox_hf = tidal_attn_prox.reindex(sdf.index, method="ffill")

        # merge
        sdf["tide"] = tide.reindex(sdf.index)
        df2 = sdf.join(tidal_hf)
        # df3 = df2.join(tidal_hf)
        # df4 = df3.join(tidal_prox_hf)
        df4 = df2.reset_index().sort_values("timestamp_utc")
        out_segments.append(df4)

    all_segments[region] = out_segments

In [ ]:
region = 11
index = 11
s = all_segments[region][index]

# Ensure datetime index
s = s.copy()
s["timestamp_utc"] = pd.to_datetime(s["timestamp_utc"], utc=True, errors="coerce")
s = s.dropna(subset=["timestamp_utc"]).set_index("timestamp_utc").sort_index()

# start = datetime(year=2020, month=8, day=1, tzinfo=UTC)
# end = datetime(year=2021, month=11, day=1, tzinfo=UTC)
# s = s[start:end]
start = s.index.min()
end = s.index.max()

s = s.resample("6h").mean()

fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(10, 4))

# Tidal power _____________________________
ax1 = axes
ax2 = ax1.twinx()

# Height (left axis)
ax1.plot(s.index, s["height"], color="tab:green", linewidth=1.2, label="Water Level", alpha=0.7)
ax1.set_ylabel("Water Level (m)")
ax1.grid(True, alpha=0.3)

# # Tide (left axis)
# tide = s["tide"]
# tide -= tide.min()
# ax1.plot(s.index, tide, color="tab:red", linewidth=1, label="tide", alpha=0.7)
# ax1.grid(True, alpha=0.3)

# Tidal power fraction (right axis)
key = "tidal_attenuation"
ax2.plot(s.index, s[key], color="tab:purple", linewidth=1.5, label="Tidal Attenuation")
ax2.set_ylabel("Tidal Attenuation/Openness")
ax2.set_ylim(-0.05, 1.05)

B = full.reset_index()
d = (
    B[
        (B.region == region)
        & (B.date.dt.date.between(start.date(), end.date()))
        & (B.source_tif.notna())
    ]
    .copy()
    .sort_values("date")
)
t = d.date

color_map = {
    "p_closed": "tab:orange",
    "p_open": "tab:blue",
}

# ax.plot(t, d[c], color=color_map.get(c, None), linewidth=2, alpha=0.9, label=c)
ax2.scatter(t, d["p_open"], color="tab:blue", alpha=0.9, label="prob open", s=10)

color_map = {
    "p_open": "tab:blue",
    "p_closed": "tab:orange",
}

pcols = ["p_open", "p_closed"]

dominant = d[pcols].idxmax(axis=1)

for col, color in color_map.items():
    mask = (dominant == col).to_numpy()
    dates = pd.to_datetime(d.date).to_numpy()
    cc = col.split("_")[1]

    seen = False
    for i in range(len(dates) - 1):
        if mask[i]:
            if not seen:
                label = f"{cc} window"
                seen = True
            else:
                label = None
            ax2.axvspan(dates[i], dates[i + 1], color=color, alpha=0.08, ec=None, label=label)

ax1.set_xlabel("Time (UTC)")

event_date = pd.Timestamp("2021-02-25", tz="UTC")

ax1.axvline(
    event_date,
    color="black",
    linestyle="--",
    linewidth=1.5,
)
ax1.text(
    event_date,
    ax1.get_ylim()[1],
    "A",
    ha="center",
    va="bottom",
    fontsize=11,
)

event_date = pd.Timestamp("2021-04-19", tz="UTC")

ax1.axvline(
    event_date,
    color="black",
    linestyle="--",
    linewidth=1.5,
)
ax1.text(
    event_date,
    ax1.get_ylim()[1],
    "B",
    ha="center",
    va="bottom",
    fontsize=11,
)

event_date = pd.Timestamp("2021-06-08", tz="UTC")

ax1.axvline(
    event_date,
    color="black",
    linestyle="--",
    linewidth=1.5,
)
ax1.text(
    event_date,
    ax1.get_ylim()[1],
    "C",
    ha="center",
    va="bottom",
    fontsize=11,
)

# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="lower left")

import matplotlib.dates as mdates

locator = mdates.AutoDateLocator()
ax1.xaxis.set_major_locator(locator)


def custom_date_formatter(x, pos):
    d = mdates.num2date(x)
    if pos == 0:
        return d.strftime("%Y-%m-%d")
    return d.strftime("%m-%d")


ax1.xaxis.set_major_formatter(plt.FuncFormatter(custom_date_formatter))

plt.tight_layout()
plt.savefig(FIG_DIR / f"{region}_example_height_tidal_attn_tide.png")
plt.show()

In [ ]:
a = "/Volumes/x10pro/estuary/sat_data/dove/results/2021/2/11/files/20210225_185139_66_105e_3B_AnalyticMS_SR_clip.tif"
b = "/Volumes/x10pro/estuary/sat_data/superdove/results/2021/4/11/files/20210419_183514_04_2307_3B_AnalyticMS_SR_8b_clip.tif"
c = "/Volumes/x10pro/estuary/sat_data/dove/results/2021/6/11/files/20210608_185607_67_105e_3B_AnalyticMS_SR_clip.tif"

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12, 4.5))

with rasterio.open(a) as src:
    data = src.read().astype(np.float32)
    nodata = src.read(1, masked=True).mask
    img = false_color(data, nodata)
    axes[0].axis("off")
    axes[0].set_title("A: 2021-02-25")
    axes[0].imshow(img)

with rasterio.open(b) as src:
    data = src.read().astype(np.float32)
    nodata = src.read(1, masked=True).mask
    img = false_color(data, nodata)
    axes[1].axis("off")
    axes[1].set_title("B: 2021-04-19")
    axes[1].imshow(img)

with rasterio.open(c) as src:
    data = src.read().astype(np.float32)
    nodata = src.read(1, masked=True).mask
    img = false_color(data, nodata)
    axes[2].axis("off")
    axes[2].set_title("C: 2021-06-08")
    axes[2].imshow(img)

fig.tight_layout()
fig.savefig(FIG_DIR / "11_example_for_timeseries.png")
plt.show()

In [ ]:
region = 59
idx = 1
s = all_segments[region][idx]

# Ensure datetime index
s = s.copy()
s["timestamp_utc"] = pd.to_datetime(s["timestamp_utc"], utc=True, errors="coerce")
s = s.dropna(subset=["timestamp_utc"]).set_index("timestamp_utc").sort_index()

start = datetime(year=2021, month=12, day=31, tzinfo=UTC)
end = datetime(year=2022, month=2, day=1, tzinfo=UTC)
s = s[start:end]
start = s.index.min()
end = s.index.max()

s = s.resample("3h").mean()

fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(10, 4))

# Tidal power _____________________________
ax1 = axes
ax2 = ax1.twinx()

# Height (left axis)
ax1.plot(s.index, s["height"], color="tab:green", linewidth=1.2, label="Water Level", alpha=0.7)
ax1.set_ylabel("Water Level (m)")
ax1.grid(True, alpha=0.3)

# Tide (left axis)
# tide = s["tide"]
# # tide -= tide.min()
# ax1.plot(s.index, tide, color="tab:red", linewidth=1, label="tide", alpha=0.7)
# ax1.grid(True, alpha=0.3)

# Tidal power fraction (right axis)
key = "tidal_attenuation"
ax2.plot(s.index, s[key], color="tab:purple", linewidth=1.5, label="Tidal Attenuation")
ax2.set_ylabel("Tidal Attenuation/Openness")
ax2.set_ylim(-0.05, 1.05)

B = full.reset_index()
d = (
    B[
        (B.region == region)
        & (B.date.dt.date.between(start.date(), end.date()))
        & (B.source_tif.notna())
    ]
    .copy()
    .sort_values("date")
)
t = d.date

color_map = {
    "p_closed": "tab:orange",
    "p_open": "tab:blue",
}

# ax.plot(t, d[c], color=color_map.get(c, None), linewidth=2, alpha=0.9, label=c)
ax2.scatter(t, d["p_open"], color="tab:blue", alpha=0.9, label="prob open", s=10)

color_map = {
    "p_open": "tab:blue",
    "p_closed": "tab:orange",
}

pcols = ["p_open", "p_closed"]

dominant = d[pcols].idxmax(axis=1)

for col, color in color_map.items():
    mask = (dominant == col).to_numpy()
    dates = pd.to_datetime(d.date).to_numpy()
    cc = col.split("_")[1]

    seen = False
    for i in range(len(dates) - 1):
        if mask[i]:
            if not seen:
                label = f"{cc} window"
                seen = True
            else:
                label = None
            ax2.axvspan(dates[i], dates[i + 1], color=color, alpha=0.08, ec=None, label=label)

ax1.set_xlabel("Time (UTC)")

event_date = pd.Timestamp("2022-01-08", tz="UTC")

ax1.axvline(
    event_date,
    color="black",
    linestyle="--",
    linewidth=1.5,
)
ax1.text(
    event_date,
    ax1.get_ylim()[1],
    "A",
    ha="center",
    va="bottom",
    fontsize=11,
)

event_date = pd.Timestamp("2022-01-10", tz="UTC")

ax1.axvline(
    event_date,
    color="black",
    linestyle="--",
    linewidth=1.5,
)
ax1.text(
    event_date,
    ax1.get_ylim()[1],
    "B",
    ha="center",
    va="bottom",
    fontsize=11,
)

event_date = pd.Timestamp("2022-01-28", tz="UTC")

ax1.axvline(
    event_date,
    color="black",
    linestyle="--",
    linewidth=1.5,
)
ax1.text(
    event_date,
    ax1.get_ylim()[1],
    "C",
    ha="center",
    va="bottom",
    fontsize=11,
)

# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="center left")

import matplotlib.dates as mdates

locator = mdates.AutoDateLocator()
ax1.xaxis.set_major_locator(locator)


def custom_date_formatter(x, pos):
    d = mdates.num2date(x)
    if pos == 0:
        return d.strftime("%Y-%m-%d")
    return d.strftime("%m-%d")


ax1.xaxis.set_major_formatter(plt.FuncFormatter(custom_date_formatter))

plt.tight_layout()
plt.savefig(FIG_DIR / f"{region}_example_height_tidal_attn_tide.png")
plt.show()

In [ ]:
a = "/Volumes/x10pro/estuary/sat_data/superdove/results/2022/1/59/files/20220108_180233_95_241f_3B_AnalyticMS_SR_8b_clip.tif"
b = "/Volumes/x10pro/estuary/sat_data/superdove/results/2022/1/59/files/20220110_180152_94_2448_3B_AnalyticMS_SR_8b_clip.tif"
c = "/Volumes/x10pro/estuary/sat_data/dove/results/2022/1/59/files/20220128_182623_1014_3B_AnalyticMS_SR_clip.tif"

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12, 4.5))

with rasterio.open(a) as src:
    data = src.read().astype(np.float32)
    nodata = src.read(1, masked=True).mask
    img = false_color(data, nodata)
    axes[0].axis("off")
    axes[0].set_title("A: 2022-01-08")
    axes[0].imshow(img)

with rasterio.open(b) as src:
    data = src.read().astype(np.float32)
    nodata = src.read(1, masked=True).mask
    img = false_color(data, nodata)
    axes[1].axis("off")
    axes[1].set_title("B: 2022-01-10")
    axes[1].imshow(img)

with rasterio.open(c) as src:
    data = src.read().astype(np.float32)
    nodata = src.read(1, masked=True).mask
    img = false_color(data, nodata)
    axes[2].axis("off")
    axes[2].set_title("C: 2022-01-28")
    axes[2].imshow(img)

fig.tight_layout()
fig.savefig(FIG_DIR / "59_example_for_timeseries.png")
plt.show()

In [ ]:
targets = full.index.to_frame(index=False).copy()

VALUE_COL = "height"
TS_COL = "timestamp_utc"
TOL = "1h"

colors = {
    1: "tab:blue",
    0: "tab:orange",
}

for val_col in ["height", "height_z", "tidal_attenuation", "tide"]:
    # We'll fill this
    targets[val_col] = pd.NA

    # Now fill targets region by region
    for region, g in tqdm.tqdm(
        targets.groupby("region", sort=False), total=len(targets.region.unique())
    ):
        if region not in all_segments:
            continue

        # Work on a view we can update, then write back
        idx = g.index
        need = targets.loc[idx, val_col].isna()
        if not need.any():
            continue

        # Each segment corresponds to ONE logger for some contiguous interval.
        # We’ll try them in priority order and fill the first that overlaps.
        for seg in all_segments[region]:
            s = seg.sort_values(TS_COL).set_index(TS_COL).resample("1D", origin="epoch").mean()
            s.index = s.index.date
            s.index = pd.to_datetime(s.index, errors="coerce")

            # Segment coverage interval
            seg_start = pd.Timestamp(s.index.min())
            seg_end = pd.Timestamp(s.index.max())

            # Which target rows still need filling AND are within this segment’s time span?
            need = targets.loc[idx, val_col].isna()
            if not need.any():
                break

            in_range = (
                need
                & (targets.loc[idx, "date"] >= seg_start)
                & (targets.loc[idx, "date"] <= seg_end)
            )
            if not in_range.any():
                continue

            # Indices of target rows we want to fill for this segment
            in_range_idx = idx[in_range]  # original targets indices (not 0..n)

            # Build left table and carry original row index explicitly
            left = targets.loc[in_range_idx, ["date"]].copy()
            left["_row"] = left.index
            left = left.sort_values("date")

            matched = pd.merge_asof(
                left,
                s[val_col],
                left_on="date",
                right_index=True,
                direction="nearest",
                # tolerance=TOL,
            )

            got = matched[val_col].notna()
            if got.any():
                rows_to_fill = matched.loc[got, "_row"].to_numpy()
                vals_to_fill = matched.loc[got, val_col].to_numpy()

                targets.loc[rows_to_fill, val_col] = vals_to_fill

    full[val_col] = targets[val_col].to_numpy()

In [ ]:
import math

import matplotlib.pyplot as plt

region_level = "region"
K = [0, 1]
label_map = {0: "closed", 1: "open"}
colors = {
    1: "tab:blue",
    0: "tab:orange",
}

df = full[(full["height"].notna()) & (full["source_tif"].notna())].copy()

regions = sorted(df.index.get_level_values(region_level).unique().tolist())

n_regions = len(regions)
ncols = 3
nrows = math.ceil(n_regions / ncols)

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(14, 2.5 * nrows),
    sharex=True,
    sharey=True,
)

# Ensure axes is always 2D
if nrows == 1:
    axes = [axes]

axes = axes.flatten()

for i, region in enumerate(regions):
    ax = axes[i]

    df_r = df.xs(region, level=region_level)

    for k in K:
        s = df_r.loc[df_r["y_pred"] == k, "tidal_attenuation"].dropna()

        if len(s) <= 1:
            continue
        if s.max() < 0.001:
            continue

        s = s.clip(0, 1.5)

        s.plot(
            kind="kde",
            ax=ax,
            color=colors[k],
            linewidth=1.6,
            alpha=0.9,
            bw_method=0.3,
        )

    name = gdf.set_index("region").loc[region]["Estuary_Name"] + " - " + str(region)
    ax.set_title(name, loc="left", fontsize=10)
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(0, 15)
    ax.grid(True, alpha=0.25)

# Turn off unused axes
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.supxlabel("Tidal Attenuation")
# fig.supylabel("Density")

plt.tight_layout()
plt.savefig(FIG_DIR / "per_estuary_tidal_attn_density.png")
plt.show()

In [ ]:
import math

import matplotlib.pyplot as plt

region_level = "region"
K = [0, 1]
label_map = {0: "closed", 1: "open"}
colors = {
    1: "tab:blue",
    0: "tab:orange",
}

df = full[(full["height"].notna()) & (full["source_tif"].notna())].copy()

# Get regions from the MultiIndex
regions = df.index.get_level_values(region_level).unique().tolist()
regions = sorted(regions)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))

for k in K:
    s_all = df.loc[df["y_pred"] == k, "tidal_attenuation"].dropna()

    s_all.plot(kind="kde", ax=ax, color=colors[k], label=label_map[k], linewidth=2, bw_method=0.3)
    # ax.hist(
    #     s_all,
    #     bins=10,
    #     density=True,
    #     alpha=0.3,
    #     color=colors[k],
    #     label=label_map[k]
    # )

ax.set_title("Tidal Attenuation Density by Predicted Class")
ax.set_xlabel("Tidal Attenuation")
ax.set_ylabel("Density")
ax.legend(frameon=False)
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(0, 13)

fig.tight_layout()
plt.savefig(FIG_DIR / "tidal_attn_by_predicted_class.png")
plt.show()

In [ ]:
import math

import matplotlib.pyplot as plt

key = "height_z"

region_level = "region"
K = [0, 1]
label_map = {0: "closed", 1: "open"}
colors = {
    1: "tab:blue",
    0: "tab:orange",
}

df = full[(full[key].notna()) & (full["source_tif"].notna())].copy()

# Get regions from the MultiIndex
regions = df.index.get_level_values(region_level).unique().tolist()
regions = sorted(regions)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))

for k in K:
    s_all = df.loc[df["y_pred"] == k, key].dropna()

    s_all.plot(kind="kde", ax=ax, color=colors[k], label=label_map[k], linewidth=2, bw_method=0.3)
    # ax.hist(
    #     s_all,
    #     bins=30,
    #     density=True,
    #     alpha=0.3,
    #     color=colors[k],
    #     label=label_map[k]
    # )

ax.set_title("Normalized Water Level by Predicted Class")
ax.set_xlabel("Normalized Water Level")
ax.set_ylabel("Density")
ax.legend(frameon=False)
# ax.set_xlim(-0.1, 1)
# ax.set_ylim(0, 10)

fig.tight_layout()
plt.savefig(FIG_DIR / "water_level_density_by_predicted_class.png")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

# -----------------------------------
# Prepare data
# -----------------------------------
df = full[full["height"].notna() & full["source_tif"].notna()].copy()

y_true = df["y_pred"].to_numpy()
score = 1.0 - df["tidal_attenuation"].astype(np.float32).to_numpy()  # higher = more open

# -----------------------------------
# ROC curve + AUC
# -----------------------------------
fpr, tpr, _ = roc_curve(y_true, score)
auc = roc_auc_score(y_true, score)
thresh = 0.96

# rule: open if tidal_attenuation < thresh
y_pred_rule = (df["tidal_attenuation"].to_numpy() < thresh).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred_rule).ravel()
fpr_point = fp / (fp + tn)
tpr_point = tp / (tp + fn)

# -----------------------------------
# Plot setup
# -----------------------------------
K = [0, 1]
label_map = {0: "closed", 1: "open"}
colors = {
    1: "tab:blue",
    0: "tab:orange",
}

fig, (ax_left, ax_right) = plt.subplots(
    1,
    2,
    figsize=(10, 4),
    gridspec_kw={"width_ratios": [2, 1]},
)

# -----------------------------------
# Left panel: tidal attenuation density by predicted class
# -----------------------------------
for k in K:
    s_all = df.loc[df["y_pred"] == k, "tidal_attenuation"].dropna()
    s_all.plot(
        kind="kde",
        ax=ax_left,
        color=colors[k],
        label=label_map[k],
        linewidth=2,
        bw_method=0.3,
    )

ax_left.set_title("Tidal Attenuation by Predicted Class")
ax_left.set_xlabel("Tidal Attenuation")
ax_left.set_ylabel("Density")
ax_left.legend(frameon=False)
ax_left.set_xlim(-0.1, 1.1)
ax_left.set_ylim(0, 13)

# -----------------------------------
# Right panel: ROC curve
# -----------------------------------
ax_right.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc:.2f}")
ax_right.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)

ax_right.scatter(
    fpr_point,
    tpr_point,
    color="red",
    s=70,
    zorder=3,
    label=f"Threshold = {thresh:.2f}",
)

ax_right.set_xlabel("False Positive Rate")
ax_right.set_ylabel("True Positive Rate")
ax_right.set_title("ROC Curve")
ax_right.legend(loc="lower right", frameon=False)

fig.tight_layout()
plt.savefig(FIG_DIR / "tidal_attn_and_roc_combined.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

# -----------------------------------
# Prepare data
# -----------------------------------
df = full[full["height"].notna() & full["source_tif"].notna()].copy()

y_true = df["y_pred"].to_numpy()
score = 1.0 - df["tidal_attenuation"].astype(np.float32).to_numpy()  # higher = more open

# -----------------------------------
# ROC curve + AUC
# -----------------------------------
fpr, tpr, _ = roc_curve(y_true, score)
auc = roc_auc_score(y_true, score)
thresh = 0.96

# -----------------------------------
# Chosen threshold operating point
# rule: open if tidal_attenuation < thresh
# -----------------------------------
y_pred_rule = (df["tidal_attenuation"].to_numpy() < thresh).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred_rule).ravel()
fpr_point = fp / (fp + tn)
tpr_point = tp / (tp + fn)

# -----------------------------------
# Plot
# -----------------------------------
fig, ax = plt.subplots(figsize=(5, 5))

ax.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc:.2f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)

ax.scatter(
    fpr_point,
    tpr_point,
    color="red",
    s=70,
    zorder=3,
    label=f"Threshold = {thresh:.2f}",
)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve: Tidal Attenuation vs Connectivity")
ax.legend(loc="lower right")

plt.tight_layout()
plt.savefig(FIG_DIR / "ROC_tidal_attn_vs_y_pred")
plt.show()

import math

import matplotlib.pyplot as plt

region_level = "region"
K = [0, 1]
label_map = {0: "closed", 1: "open"}
colors = {
    1: "tab:blue",
    0: "tab:orange",
}

df = full[(full["height"].notna()) & (full["source_tif"].notna())].copy()

# Get regions from the MultiIndex
regions = df.index.get_level_values(region_level).unique().tolist()
regions = sorted(regions)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))

for k in K:
    s_all = df.loc[df["y_pred"] == k, "tidal_attenuation"].dropna()

    s_all.plot(kind="kde", ax=ax, color=colors[k], label=label_map[k], linewidth=2, bw_method=0.3)
    # ax.hist(
    #     s_all,
    #     bins=10,
    #     density=True,
    #     alpha=0.3,
    #     color=colors[k],
    #     label=label_map[k]
    # )

ax.set_title("Tidal Attenuation Density by Predicted Class")
ax.set_xlabel("Tidal Attenuation")
ax.set_ylabel("Density")
ax.legend(frameon=False)
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(0, 13)

fig.tight_layout()
plt.savefig(FIG_DIR / "tidal_attn_by_predicted_class.png")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


def plot_region_probs_and_tp(
    full,
    to_plot,
    save_path,
    *,
    region_level="region",
    date_level="date",  # change if your level name differs
    prob_cols=("p_closed", "p_open"),
):
    if len(to_plot) == 0:
        return

    # fig, axes = plt.subplots(nrows=len(to_plot), ncols=2, figsize=(12, 5 * len(to_plot)))
    fig, axes = plt.subplots(
        nrows=len(to_plot),
        ncols=2,
        figsize=(14, 5 * len(to_plot)),
        gridspec_kw={"width_ratios": [2, 1]},
    )

    if len(to_plot) == 1:
        axes = [axes]

    for i, (_, row) in enumerate(to_plot.iterrows()):
        region = row.region
        start = row.date - pd.Timedelta("14D")
        end = row.date + pd.Timedelta("14D")
        start_utc = start.tz_localize("UTC")
        end_utc = end.tz_localize("UTC")
        source_tif = row.source_tif
        axs = axes[i]

        # Ensure timestamps
        start = pd.Timestamp(start)
        end = pd.Timestamp(end)

        B = full.reset_index()
        d = B[(B.region == region) & (B.date.between(start, end))].copy().sort_values("date")
        t = d.date

        if d.empty:
            raise ValueError("No rows after filtering. Check region/start/end and level names.")

        # --- plot ---
        ax = axs[0]
        img_ax = axs[1]
        ax2 = ax.twinx()

        color_map = {
            "p_closed": "tab:orange",
            "p_open": "tab:blue",
        }

        # Probabilities (left axis)
        for c in prob_cols:
            if c not in d.columns:
                continue
            # ax.plot(t, d[c], color=color_map.get(c, None), linewidth=2, alpha=0.9, label=c)
            dd = d[d.source_tif.notna()]
            y = dd[c]
            tt = dd["date"]
            ax.scatter(tt, y, color=color_map.get(c, None), alpha=0.9, label=f"{c} sample")

        for df in all_segments[region]:
            dd = df.copy()
            dd["timestamp_utc"] = pd.to_datetime(dd["timestamp_utc"], utc=True, errors="coerce")
            dd = dd.dropna(subset=["timestamp_utc"]).sort_values("timestamp_utc")
            dd = subsample_time(dd, "1h")

            dd = dd[(dd["timestamp_utc"] >= start_utc) & (dd["timestamp_utc"] < end_utc)]
            if dd.empty:
                continue

            tt = dd["timestamp_utc"]

            h = dd["height"]
            ax2.plot(tt, h, color="blue", linewidth=1.5, alpha=0.5, label="height")

            h = dd["tide"]
            ax2.plot(tt, h, color="purple", linewidth=1.5, alpha=0.5, label="tide")

            # range
            trange = dd["tidal_attenuation"].astype(float)
            ax.plot(tt, trange, color="maroon", linewidth=1.5, alpha=0.8, label="tidal ratio")

        ax.set_xlabel("Time")
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

        ax.axvline(row.date, linestyle=":", color="black", linewidth=2)
        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel("Model probability/Tidal Attenuation")
        ax.grid(True, alpha=0.3)
        if i == 0:
            name = gdf.set_index("region").loc[region]["Estuary_Name"]
            ax.set_title(f"{name} - {region} - {row.date}")
        else:
            ax.set_title(row.date)

        # Combined legend
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(
            h1 + h2,
            l1 + l2,
            loc="upper right",
            bbox_to_anchor=(-0.02, 1),  # move outside to the left
            borderaxespad=0,
            frameon=False,
        )

        ax2.set_ylabel("Water/Tide Height (m)")

        with rasterio.open(source_tif) as src:
            data = src.read()
            nodata = src.read(1, masked=True).mask
            img = false_color(data, nodata)
        img_ax.axis("off")
        img_ax.imshow(img)

    plt.tight_layout(rect=[0.08, 0, 1, 1])
    plt.savefig(save_path)
    plt.close()
    # plt.show()


def even_time_sample(df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    df = df.sort_values("date").reset_index(drop=True)

    if len(df) <= n:
        return df

    # Evenly spaced index positions
    idx = np.linspace(0, len(df) - 1, n, dtype=int)
    return df.iloc[idx]

In [ ]:
A = full.reset_index()
aa = A[A.height.notna()].region.unique()
for region in tqdm.tqdm(aa):
    inspect = A[
        (A.region == region)
        & (A.source_tif.notna())
        & (A.height.notna())
        & (A.tidal_attenuation > 0.8)
        & (A.y_pred == 1)
    ].copy()
    inspect = even_time_sample(inspect)

    plot_region_probs_and_tp(
        full, to_plot=inspect, save_path=FIG_DIR / f"open_tidal_ratio_inspect_{region}.png"
    )

print("Done")

In [ ]:
A = full.reset_index()
aa = A[A.height.notna()].region.unique()
for region in tqdm.tqdm(aa):
    inspect = A[
        (A.region == region)
        & (A.source_tif.notna())
        & (A.height.notna())
        & (A.tidal_attenuation < 0.8)
        & (A.y_pred == 0)
    ].copy()
    inspect = even_time_sample(inspect)

    plot_region_probs_and_tp(
        full, to_plot=inspect, save_path=FIG_DIR / f"closed_tidal_ratio_inspect_{region}.png"
    )

print("Done")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


def plot_single_region_probs_and_tp(
    full,
    region,
    date,
    axs,
    include_legend=False,
):
    date = pd.Timestamp(date)    
    start = date - pd.Timedelta("14D")
    end = date + pd.Timedelta("14D")
    start_utc = start.tz_localize("UTC")
    end_utc = end.tz_localize("UTC")

    source_tif = full.loc[region, date].source_tif

    B = full.reset_index()
    d = B[(B.region == region) & (B.date.between(start, end))].copy().sort_values("date")
    t = d.date

    if d.empty:
        raise ValueError("No rows after filtering. Check region/start/end and level names.")

    # --- plot ---
    ax = axs[0]
    img_ax = axs[1]
    ax2 = ax.twinx()

    color_map = {
        "p_open": "tab:blue",
    }

    dd = d[d.source_tif.notna()]
    y = dd["p_open"]
    tt = dd["date"]
    ax.scatter(tt, y, color=color_map.get("p_open", None), alpha=0.9, label=f"p_open")

    for df in all_segments[region]:
        dd = df.copy()
        dd["timestamp_utc"] = pd.to_datetime(dd["timestamp_utc"], utc=True, errors="coerce")
        dd = dd.dropna(subset=["timestamp_utc"]).sort_values("timestamp_utc")
        dd = subsample_time(dd, "1h")

        dd = dd[(dd["timestamp_utc"] >= start_utc) & (dd["timestamp_utc"] < end_utc)]
        if dd.empty:
            continue

        tt = dd["timestamp_utc"]

        h = dd["height"]
        ax2.plot(tt, h, color="blue", linewidth=1.5, alpha=0.5, label="Water Level")

        h = dd["tide"]
        ax2.plot(tt, h, color="purple", linewidth=1.5, alpha=0.5, label="Tide Height")

        # range
        trange = dd["tidal_attenuation"].astype(float)
        ax.plot(tt, trange, color="maroon", linewidth=1.5, alpha=0.8, label="Tidal Attn.")

    ax.set_xlabel("Time")
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))

    ax.axvline(date, linestyle=":", color="black", linewidth=1.5)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("Model probability/Tidal Attenuation")
    ax2.set_ylabel("Water/Tide Height (m)")

    if include_legend:
        # Combined legend
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(
            h1 + h2,
            l1 + l2,
            loc="upper right",
            bbox_to_anchor=(-0.1, 1),  # move outside to the left
            borderaxespad=0,
            frameon=False,
        )

    with rasterio.open(source_tif) as src:
        data = src.read()
        nodata = src.read(1, masked=True).mask
        img = false_color(data, nodata)
    img_ax.axis("off")
    img_ax.imshow(img)
    name = gdf.set_index("region").loc[region]["Estuary_Name"]
    img_ax.set_title(f"{name} - {date.date()}")


rows = 5
fig, axes = plt.subplots(
    nrows=rows,
    ncols=2,
    figsize=(14, 5 * rows),
    gridspec_kw={"width_ratios": [2, 1]},
)
plot_single_region_probs_and_tp(
    full, 
    region=48,
    date=pd.Timestamp(year=2018, month=2, day=3),
    axs=axes[0],
    include_legend=True,
)
plot_single_region_probs_and_tp(
    full, 
    region=72,
    date=pd.Timestamp(year=2023, month=10, day=16),
    axs=axes[1],
)
plot_single_region_probs_and_tp(
    full, 
    region=11,
    date=pd.Timestamp(year=2022, month=5, day=5),
    axs=axes[2],
)
plot_single_region_probs_and_tp(
    full, 
    region=28,
    date=pd.Timestamp(year=2022, month=2, day=22),
    axs=axes[3],
)
plot_single_region_probs_and_tp(
    full, 
    region=50,
    date=pd.Timestamp(year=2022, month=2, day=17),
    axs=axes[4],
)
fig.tight_layout()
plt.savefig(FIG_DIR / "examples_open_low_tidal.png", dpi=300)
plt.show()

rows = 3
fig, axes = plt.subplots(
    nrows=rows,
    ncols=2,
    figsize=(14, 5 * rows),
    gridspec_kw={"width_ratios": [2, 1]},
)
plot_single_region_probs_and_tp(
    full, 
    region=43,
    date=pd.Timestamp(year=2024, month=5, day=8),
    axs=axes[0],
    include_legend=True,
)
plot_single_region_probs_and_tp(
    full, 
    region=59,
    date=pd.Timestamp(year=2018, month=2, day=25),
    axs=axes[1],
)
plot_single_region_probs_and_tp(
    full, 
    region=48,
    date=pd.Timestamp(year=2020, month=2, day=15),
    axs=axes[2],
)
fig.tight_layout()
plt.savefig(FIG_DIR / "examples_closed_high_tidal.png", dpi=300)
plt.show()